# CSAM Ablation Study & Grid Search — Colab TPU v5e-1

**Cognitive Sparse Access Memory (CSAM)** — 3-tier hierarchical memory for AI agents.

This notebook runs two evaluations end-to-end:
1. **Ablation v3** — Compares 4 forgetting strategies (No-Forgetting, LRU, Importance, Consolidation-Aware) using real LLM QA via Groq
2. **Grid Search** — Tests 20 weight combinations (α, β, γ, δ) for the CA forgetting formula

**Runtime:** TPU v5e-1 (selected for high RAM and fast CPU; embedding model runs on CPU/TPU host)  
**API:** Groq free tier (llama-3.1-8b-instant) with 2-key rotation  
**Estimated time:** ~45–60 min total

## 1. Check Runtime Environment & TPU Availability

Verify we’re running on a Colab TPU runtime and inspect the available hardware.

In [ ]:
import os
import platform

print("=" * 60)
print("RUNTIME ENVIRONMENT CHECK")
print("=" * 60)

# Check if we're on Colab
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
print(f"Running on Colab:  {IN_COLAB}")
print(f"Python:            {platform.python_version()}")
print(f"Platform:          {platform.platform()}")

# Check TPU availability
tpu_name = os.environ.get("TPU_NAME", None)
colab_tpu = os.environ.get("COLAB_TPU_ADDR", None)
print(f"TPU_NAME:          {tpu_name or 'not set'}")
print(f"COLAB_TPU_ADDR:    {colab_tpu or 'not set'}")

# RAM info
import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System RAM:        {ram_gb:.1f} GB")

if tpu_name or colab_tpu:
    print("\n\u2705 TPU v5e-1 runtime detected")
else:
    print("\n\u26a0\ufe0f  No TPU env vars found - may be using CPU/GPU runtime")
    print("    (CSAM workload is CPU + API-bound, will still work fine)")
print("=" * 60)

## 2. Install Required Dependencies

Install the Python packages needed for CSAM. The TPU v5e-1 host provides high RAM for embedding model loading.

In [ ]:
# Core CSAM dependencies
!pip install -q python-dotenv sentence-transformers hnswlib

# JAX TPU support (for future TPU-accelerated embedding)
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

print("\n\u2705 All dependencies installed")

## 3. Clone Repository & Configure TPU Device

Clone the CSAM repo from GitHub (main branch), enumerate TPU/JAX devices, and confirm connectivity.

In [ ]:
import os
import shutil

# Clone CSAM repo (main branch)
REPO_URL = "https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git"
REPO_DIR = "/content/CSAM-IPD-HALH"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch main --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"\n\U0001f4c2 Working directory: {os.getcwd()}")

# Verify JAX TPU device mapping
import jax
devices = jax.devices()
print(f"\nJAX backend:       {jax.default_backend()}")
print(f"JAX devices found: {len(devices)}")
for i, d in enumerate(devices):
    print(f"  [{i}] {d.device_kind} - {d}")

tpu_devices = [d for d in devices if "tpu" in d.device_kind.lower()]
if tpu_devices:
    print(f"\n\u2705 {len(tpu_devices)} TPU core(s) available - mesh layout ready")
else:
    print("\n\u26a0\ufe0f  No TPU cores via JAX (CPU-only fallback - CSAM still works fine)")

## 4. Set API Keys & Verify Groq Connection

Load Groq API keys from Colab Secrets. Go to the **🔑 Secrets** panel (key icon in left sidebar), add:
- `GROQ_API_KEY` — your primary Groq key
- `GROQ_API_KEY_2` — your second Groq key (for rate-limit rotation)

In [ ]:
import os

# Load API keys from Colab Secrets (preferred - no hardcoded keys)
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    os.environ["GROQ_API_KEY_2"] = userdata.get("GROQ_API_KEY_2")
    print("\u2705 API keys loaded from Colab Secrets")
except Exception:
    # Fallback: check if already set in environment
    if "GROQ_API_KEY" in os.environ:
        print("\u2705 API keys found in environment")
    else:
        raise RuntimeError(
            "\u274c No API keys found! Add GROQ_API_KEY to Colab Secrets "
            "(key icon in left sidebar)"
        )

# Write .env file for the CSAM scripts (they use python-dotenv)
env_lines = []
for key in ["GROQ_API_KEY", "GROQ_API_KEY_2"]:
    val = os.environ.get(key, "")
    if val:
        env_lines.append(f"{key}={val}")

with open(".env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

print(f"\U0001f4dd .env written with {len(env_lines)} key(s)")

# Quick Groq connectivity check
import requests
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=10,
)
if resp.status_code == 200:
    models = [m["id"] for m in resp.json().get("data", [])]
    print(f"\u2705 Groq API connected - {len(models)} models available")
    if "llama-3.1-8b-instant" in models:
        print("   \u2713 llama-3.1-8b-instant is available")
else:
    print(f"\u274c Groq API error: {resp.status_code} - check your key")

## 5. Verify CSAM Module Imports & Embedding Model

Load the CSAM core modules and warm up the sentence-transformer embedding model. This confirms the full dependency chain works before running the evaluation.

In [ ]:
import sys
import os
import time

os.chdir("/content/CSAM-IPD-HALH")
sys.path.insert(0, os.path.join(os.getcwd(), "csam_project"))

# Test all critical imports
from csam_core.memory_repository import MemoryRepository
from csam_core.knowledge_graph import KnowledgeGraph
from csam_core.forgetting_engine import ConsolidationAwareForgetting
from csam_core.consolidation_tracker import ConsolidationTracker
from csam_core.consolidation import ConsolidationPipeline
from csam_core.retrieval import HybridRetriever
from csam_core.services.embedding import EmbeddingService
from csam_core.services.llm_hosted import HostedLLMService
from evaluation.npc_locomo import BenchmarkGenerator
from evaluation.run_ablation import evaluate_strategy

print("\u2705 All CSAM modules imported successfully")

# Warm up embedding model (first load takes ~10s)
print("\n\u23f3 Loading embedding model (all-MiniLM-L6-v2)...")
t0 = time.time()
emb = EmbeddingService()
dim = emb.dimension  # forces model load
test_vec = emb.encode("test query")
print(f"\u2705 Embedding model loaded in {time.time() - t0:.1f}s")
print(f"   Dimension: {dim}, test vector shape: {test_vec.shape}")

# Verify LLM service
llm = HostedLLMService(provider="groq", model="llama-3.1-8b-instant")
n_keys = len(llm._api_keys)
print(f"\n\u2705 HostedLLMService ready - {n_keys} API key(s) loaded")
print(f"   Provider: groq | Model: llama-3.1-8b-instant")

# Quick benchmark data check
gen = BenchmarkGenerator(seed=42)
ds = gen.generate_benchmark_dataset(num_conversations=1, interactions_per_conversation=10)
imps = [i.importance for i in ds[0].interactions]
print(f"\n\u2705 Benchmark generator OK - importance range: [{min(imps):.2f}, {max(imps):.2f}]")
print(f"   (Category-based heuristics, NOT random)")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED - ready to run evaluations")
print("=" * 60)

## 6. Run Ablation v3 — Fixed Importance Heuristics

Evaluates 4 forgetting strategies on the NPC-LoCoMo benchmark:
- **No-Forgetting** — unbounded memory (baseline)
- **LRU** — least recently used (baseline)
- **Importance** — forget least important first (baseline)
- **Consolidation-Aware (Ours)** — α·R + β·(1-I) + γ·C + δ·D (novel)

Configuration: 5 conversations × 100 interactions × threshold=80  
QA: Real LLM answers via Groq (llama-3.1-8b-instant)  
Expected: ~272 API calls, ~85K tokens, ~10–15 min

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

ABLATION_OUTPUT = "csam_project/evaluation/ablation_results_v3.json"

print("\U0001f680 Starting Ablation v3...")
print(f"   Output: {ABLATION_OUTPUT}")
t_start = time.time()

!python -m csam_project.evaluation.run_ablation \
    --conversations 5 \
    --interactions 100 \
    --threshold 80 \
    --output {ABLATION_OUTPUT} \
    --model llama-3.1-8b-instant

elapsed = time.time() - t_start
print(f"\n\u23f1\ufe0f  Ablation v3 completed in {elapsed/60:.1f} min")

## 7. Run Grid Search — 20 Weight Combinations

Searches over 20 strategic (α, β, γ, δ) weight combos for the CA forgetting formula.  
Only tests the Consolidation-Aware strategy per combo (~68 QA calls each).  
Expected: ~1,360 API calls, ~440K tokens, ~35–45 min

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

GRID_OUTPUT = "csam_project/evaluation/grid_search_results.json"

print("\U0001f680 Starting Grid Search (20 weight combos)...")
print(f"   Output: {GRID_OUTPUT}")
t_start = time.time()

!python -m csam_project.evaluation.grid_search_weights \
    --conversations 5 \
    --interactions 100 \
    --threshold 80 \
    --output {GRID_OUTPUT} \
    --model llama-3.1-8b-instant \
    --quiet

elapsed = time.time() - t_start
print(f"\n\u23f1\ufe0f  Grid search completed in {elapsed/60:.1f} min")

## 8. Display Results & Performance Metrics

Load both result files and display summary tables with F1 scores, timing, and the best weight configuration found by grid search.

In [ ]:
import json
import os

os.chdir("/content/CSAM-IPD-HALH")

# Ablation v3 Results
ABLATION_FILE = "csam_project/evaluation/ablation_results_v3.json"
print("=" * 70)
print("ABLATION v3 RESULTS (Fixed Importance Heuristics)")
print("=" * 70)

if os.path.exists(ABLATION_FILE):
    with open(ABLATION_FILE) as f:
        ablation = json.load(f)

    header = f"{'Strategy':<30} {'F1':>8} {'S-Hop':>8} {'M-Hop':>8} {'Temp':>8} {'Adv':>8} {'Mem':>6}"
    print(f"\n{header}")
    print("-" * 78)
    for r in ablation["results"]:
        f1s = r["f1_scores"]
        print(f"{r['strategy']:<30} "
              f"{r['overall_f1']:>8.3f} "
              f"{f1s.get('single-hop', 0):>8.3f} "
              f"{f1s.get('multi-hop', 0):>8.3f} "
              f"{f1s.get('temporal', 0):>8.3f} "
              f"{f1s.get('adversarial', 0):>8.3f} "
              f"{r['memory_count']:>6}")

    if ablation.get("api_usage"):
        u = ablation["api_usage"]
        print(f"\nAPI Usage: {u['total_requests']} requests, "
              f"{u['total_tokens']:,} tokens, "
              f"{u['avg_latency_ms']:.0f}ms avg")
else:
    print(f"\u26a0\ufe0f  {ABLATION_FILE} not found")

# Grid Search Results
GRID_FILE = "csam_project/evaluation/grid_search_results.json"
print("\n\n" + "=" * 70)
print("GRID SEARCH RESULTS (Ranked by F1)")
print("=" * 70)

if os.path.exists(GRID_FILE):
    with open(GRID_FILE) as f:
        grid = json.load(f)

    header2 = f"{'Rank':<6}{'a':>6}{'b':>6}{'g':>6}{'d':>6}  {'F1':>8}  {'S-Hop':>8}  {'Mem':>6}"
    print(f"\n{header2}")
    print("-" * 62)
    for rank, entry in enumerate(grid["results"], 1):
        w = entry["weights"]
        print(f"{rank:<6}{w['alpha']:>6.2f}{w['beta']:>6.2f}"
              f"{w['gamma']:>6.2f}{w['delta']:>6.2f}  "
              f"{entry['overall_f1']:>8.4f}  "
              f"{entry['f1_scores'].get('single-hop', 0):>8.4f}  "
              f"{entry['memory_count']:>6}")

    bw = grid["best_weights"]
    print(f"\n\u2605 BEST: a={bw['alpha']} b={bw['beta']} g={bw['gamma']} d={bw['delta']}"
          f"  ->  F1={grid['best_f1']:.4f}")

    if grid.get("api_usage"):
        u = grid["api_usage"]
        print(f"\nAPI Usage: {u['total_requests']} requests, "
              f"{u['total_tokens']:,} tokens")
else:
    print(f"\u26a0\ufe0f  {GRID_FILE} not found")

## 9. Validate Output Correctness

Run assertions to verify that result files exist, contain valid data, and the CA strategy outperforms baselines.

In [ ]:
import json
import os

os.chdir("/content/CSAM-IPD-HALH")

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  \u2705 {name}")
        passed += 1
    else:
        print(f"  \u274c {name}")
        failed += 1

print("=" * 60)
print("VALIDATION ASSERTIONS")
print("=" * 60)

# Ablation v3 checks
ABLATION_FILE = "csam_project/evaluation/ablation_results_v3.json"
print(f"\n\U0001f4cb Ablation v3 ({ABLATION_FILE}):")
check("Result file exists", os.path.exists(ABLATION_FILE))

if os.path.exists(ABLATION_FILE):
    with open(ABLATION_FILE) as f:
        ab = json.load(f)

    check("Has 4 strategy results", len(ab["results"]) == 4)
    check("Config matches (5 conv, 100 int, thresh 80)",
          ab["config"]["num_conversations"] == 5
          and ab["config"]["interactions_per_conversation"] == 100
          and ab["config"]["forget_threshold"] == 80)
    check("Used LLM QA (not word-overlap)", ab["config"]["use_llm"] is True)

    strats = {r["strategy"]: r for r in ab["results"]}
    ca_f1 = strats.get("Consolidation-Aware (Ours)", {}).get("overall_f1", 0)
    lru_f1 = strats.get("LRU", {}).get("overall_f1", 0)
    check(f"CA overall F1 > 0 (got {ca_f1:.3f})", ca_f1 > 0)
    check(f"CA F1 ({ca_f1:.3f}) >= LRU F1 ({lru_f1:.3f})", ca_f1 >= lru_f1)

    ca_sh = strats.get("Consolidation-Aware (Ours)", {}).get("f1_scores", {}).get("single-hop", 0)
    lru_sh = strats.get("LRU", {}).get("f1_scores", {}).get("single-hop", 0)
    check(f"CA single-hop ({ca_sh:.3f}) > LRU single-hop ({lru_sh:.3f})", ca_sh > lru_sh)

# Grid Search checks
GRID_FILE = "csam_project/evaluation/grid_search_results.json"
print(f"\n\U0001f4cb Grid Search ({GRID_FILE}):")
check("Result file exists", os.path.exists(GRID_FILE))

if os.path.exists(GRID_FILE):
    with open(GRID_FILE) as f:
        gs = json.load(f)

    check("Has 20 combo results", len(gs["results"]) == 20)
    check("Best F1 > 0", gs["best_f1"] > 0)

    bw = gs["best_weights"]
    weight_sum = bw["alpha"] + bw["beta"] + bw["gamma"] + bw["delta"]
    check(f"Best weights sum to 1.0 (got {weight_sum:.2f})", abs(weight_sum - 1.0) < 0.01)

    f1s = [r["overall_f1"] for r in gs["results"]]
    check("Results sorted descending by F1", f1s == sorted(f1s, reverse=True))

# Summary
print(f"\n{'=' * 60}")
status = "PASS \u2705" if failed == 0 else "FAIL \u274c"
print(f"RESULT: {status} - {passed} passed, {failed} failed")
print(f"{'=' * 60}")

## 10. Download Result Files

Download the JSON results to your local machine.

In [ ]:
import os
os.chdir("/content/CSAM-IPD-HALH")

from google.colab import files

result_files = [
    "csam_project/evaluation/ablation_results_v3.json",
    "csam_project/evaluation/grid_search_results.json",
]

for fp in result_files:
    if os.path.exists(fp):
        files.download(fp)
        print(f"\U0001f4e5 Downloaded: {fp}")
    else:
        print(f"\u26a0\ufe0f  Not found: {fp}")